# Study 2: compact environment validation

This notebook checks a hand-calculated transition, transport conservation with uneven regional values, cleanup in isolation, and a deliberately triggered pollution failure.

## 1. Set up controlled configurations

The hand-calculation case disables natural recovery and cleanup. Only region 0 has an air stock and air emissions, so the expected transport can be calculated directly. Other checks use fresh configurations.

In [38]:
from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.environment import ACTION_NAMES, EconomicSustainabilityEnvironment

In [39]:
def make_validation_config():
    config = EnvironmentConfig()
    config.baseline_emissions = {pollutant: 0.0 for pollutant in config.pollutants}
    config.edi_emissions_per_point = {pollutant: 0.0 for pollutant in config.pollutants}
    config.initial_pollution = {pollutant: 0.0 for pollutant in config.pollutants}
    config.natural_recovery_rates = {pollutant: 0.0 for pollutant in config.pollutants}
    config.recovery_degradation = {pollutant: 0.0 for pollutant in config.pollutants}

    return config

In [40]:
NO_INVESTMENT = ACTION_NAMES.index("maintain_none")
HIGH_CLEANUP = ACTION_NAMES.index("maintain_clean_high")
NO_INVESTMENT_ACTIONS = (NO_INVESTMENT,) * 4
validation_checks = []


def check_validation(condition, description):
    if condition:
        status = "PASS"
    else:
        status = "FAIL"

    validation_checks.append((status, description))
    print(f"{status}: {description}")

## 2. Compare a hand calculation with one environment step

Air stock starts at 12 in region 0 and zero elsewhere. Region 0 has EDI 4; air emissions are 1 unit per EDI point, so it emits 4. The air transport fraction is 0.30 and every region has two neighbors. No natural recovery or cleanup occurs.

Region 0 retains 70% of its old stock and emissions: `12 * 0.70 + 4 * 0.70 = 11.2`. Regions 1 and 2 each receive half of region 0's transported stock and emissions: `12 * 0.30 / 2 + 4 * 0.30 / 2 = 2.4`. Region 3 receives zero. Expected air loads: `(11.2, 2.4, 2.4, 0.0)`.

In [41]:
config = make_validation_config()
config.edi_emissions_per_point["air"] = 1.0
config.transport_fractions["air"] = 0.30
config.cleanup_capacities = (0.0, 0.0, 0.0, 0.0)

In [42]:
environment = EconomicSustainabilityEnvironment("local", config)
environment.reset(101)
environment.development_indices = (4, 0, 0, 0)
environment.pollutant_loads["air"] = (12.0, 0.0, 0.0, 0.0)
environment._update_indices()
starting_pollutant_loads = dict(environment.pollutant_loads)

In [43]:
step_result = environment.step(NO_INVESTMENT_ACTIONS)

In [44]:
observations, rewards, terminated, truncated, info = step_result
expected_air_loads = (11.2, 2.4, 2.4, 0.0)
observed_air_loads = info["pollutant_loads"]["air"]

print("Expected air loads:", expected_air_loads)
print("Observed air loads:", observed_air_loads)

Expected air loads: (11.2, 2.4, 2.4, 0.0)
Observed air loads: (11.2, 2.4, 2.4, 0.0)


In [45]:
for region in config.regions:
    difference = observed_air_loads[region] - expected_air_loads[region]
    check_validation(abs(difference) < 1e-9, f"Region {region} hand-calculated air load")

print("Hand calculation matches the environment.")

PASS: Region 0 hand-calculated air load
PASS: Region 1 hand-calculated air load
PASS: Region 2 hand-calculated air load
PASS: Region 3 hand-calculated air load
Hand calculation matches the environment.


## 3. Check transport conservation with uneven regional values

The hand-calculation case uses uneven air stocks and emissions. For every pollutant, total incoming transported emissions must equal total emissions sent out. Total incoming transported stocks must equal the transported portion of stocks remaining after natural recovery. These checks stop before cleanup, which removes pollution afterward.

In [46]:
transport_totals = {}

for pollutant in config.pollutants:
    remaining_emissions = info["remaining_emissions"][pollutant]
    fraction = config.transport_fractions[pollutant]
    outgoing_emissions = sum(remaining_emissions) * fraction
    incoming_emissions = sum(info["incoming_emissions"][pollutant])

    recovered_stocks = []
    for region in config.regions:
        old_load = starting_pollutant_loads[pollutant][region]
        removed_naturally = info["natural_removal"][pollutant][region]
        recovered_stocks.append(old_load - removed_naturally)

    outgoing_stocks = sum(recovered_stocks) * fraction
    incoming_stocks = sum(info["incoming_pollution"][pollutant])
    transport_totals[pollutant] = (outgoing_emissions, incoming_emissions, outgoing_stocks, incoming_stocks)

In [47]:
for pollutant, totals in transport_totals.items():
    outgoing_emissions, incoming_emissions, outgoing_stocks, incoming_stocks = totals

    emissions_conserved = abs(incoming_emissions - outgoing_emissions) < 1e-9
    stocks_conserved = abs(incoming_stocks - outgoing_stocks) < 1e-9
    check_validation(emissions_conserved, f"{pollutant} emissions transport conservation")
    check_validation(stocks_conserved, f"{pollutant} stock transport conservation")

    print(pollutant, totals)

print("Transported emissions and recovered stocks are conserved.")

PASS: air emissions transport conservation
PASS: air stock transport conservation
air (1.2, 1.2, 3.5999999999999996, 3.5999999999999996)
PASS: water emissions transport conservation
PASS: water stock transport conservation
water (0.0, 0.0, 0.0, 0.0)
PASS: soil emissions transport conservation
PASS: soil stock transport conservation
soil (0.0, 0.0, 0.0, 0.0)
PASS: noise emissions transport conservation
PASS: noise stock transport conservation
noise (0.0, 0.0, 0.0, 0.0)
Transported emissions and recovered stocks are conserved.


## 4. Isolate cleanup from prevention and development

Run the same starting state twice, changing only region 0's action from no investment to high cleanup. Emissions are disabled. Cleanup should be paid, remove positive pollution in region 0, and leave other regions' post-step loads unchanged.

In [48]:
cleanup_config = make_validation_config()
cleanup_config.initial_pollution = {pollutant: 20.0 for pollutant in cleanup_config.pollutants}
control_environment = EconomicSustainabilityEnvironment("local", cleanup_config)
control_environment.reset(202)

((0, 2, 0, 0, 0, 0, 0),
 (0, 1, 0, 0, 0, 0, 0),
 (0, 2, 0, 0, 0, 0, 0),
 (0, 2, 0, 0, 0, 0, 0))

In [49]:
control_result = control_environment.step(NO_INVESTMENT_ACTIONS)
control_observations, control_rewards, control_terminated, control_truncated, control_info = control_result

In [50]:
cleanup_environment = EconomicSustainabilityEnvironment("local", cleanup_config)
cleanup_environment.reset(202)
cleanup_actions = (HIGH_CLEANUP, NO_INVESTMENT, NO_INVESTMENT, NO_INVESTMENT)

In [51]:
cleanup_result = cleanup_environment.step(cleanup_actions)
cleanup_observations, cleanup_rewards, cleanup_terminated, cleanup_truncated, cleanup_info = cleanup_result

In [52]:
check_validation(cleanup_info["valid_actions"][0], "Cleanup action is affordable and valid")
cleanup_cost_is_correct = cleanup_info["cleanup_costs_paid"][0] == cleanup_config.cleanup_costs[3]
check_validation(cleanup_cost_is_correct, "High cleanup cost is recorded")

for pollutant in cleanup_config.pollutants:
    removed = cleanup_info["cleanup_removed"][pollutant][0]
    cleaned_load = cleanup_info["pollutant_loads"][pollutant][0]
    control_load = control_info["pollutant_loads"][pollutant][0]

    check_validation(removed > 0.0, f"{pollutant} cleanup removes pollution")
    check_validation(cleaned_load < control_load, f"{pollutant} load falls under cleanup")

    for region in (1, 2, 3):
        cleaned_other_region = cleanup_info["pollutant_loads"][pollutant][region]
        control_other_region = control_info["pollutant_loads"][pollutant][region]
        difference = cleaned_other_region - control_other_region
        unchanged = abs(difference) < 1e-9
        check_validation(unchanged, f"{pollutant} region {region} is unaffected")

print("Cleanup spending:", cleanup_info["cleanup_costs_paid"])
print("High cleanup reduces existing loads in the selected region only.")

PASS: Cleanup action is affordable and valid
PASS: High cleanup cost is recorded
PASS: air cleanup removes pollution
PASS: air load falls under cleanup
PASS: air region 1 is unaffected
PASS: air region 2 is unaffected
PASS: air region 3 is unaffected
PASS: water cleanup removes pollution
PASS: water load falls under cleanup
PASS: water region 1 is unaffected
PASS: water region 2 is unaffected
PASS: water region 3 is unaffected
PASS: soil cleanup removes pollution
PASS: soil load falls under cleanup
PASS: soil region 1 is unaffected
PASS: soil region 2 is unaffected
PASS: soil region 3 is unaffected
PASS: noise cleanup removes pollution
PASS: noise load falls under cleanup
PASS: noise region 1 is unaffected
PASS: noise region 2 is unaffected
PASS: noise region 3 is unaffected
Cleanup spending: (80.0, 0.0, 0.0, 0.0)
High cleanup reduces existing loads in the selected region only.


## 5. Deliberately trigger a pollution failure

Set air emissions far above the synthetic limit, then take no-investment actions. The step should terminate, identify failed regions, and apply the failure penalty to every selected reward.

In [53]:
failure_config = make_validation_config()
failure_config.initial_pollution = {pollutant: 10.0 for pollutant in failure_config.pollutants}
failure_config.baseline_emissions["air"] = 500.0
failure_environment = EconomicSustainabilityEnvironment("local", failure_config)
failure_environment.reset(303)

((0, 0, 0, 0, 0, 0, 0),
 (0, 1, 0, 0, 0, 0, 0),
 (0, 2, 0, 0, 0, 0, 0),
 (0, 1, 0, 0, 0, 0, 0))

In [54]:
failure_result = failure_environment.step(NO_INVESTMENT_ACTIONS)
failure_observations, failure_rewards, terminated, truncated, failure_info = failure_result

In [55]:
check_validation(terminated, "Pollution limit failure terminates the episode")
check_validation(not truncated, "Failure is not reported as a time-limit truncation")
check_validation(bool(failure_info["failed_regions"]), "Failed regions and pollutants are recorded")

for region, pollutant in failure_info["failed_regions"]:
    failed_load = failure_info["pollutant_loads"][pollutant][region]
    limit = failure_config.pollution_limits[pollutant]
    check_validation(failed_load >= limit, f"Region {region} {pollutant} reaches its limit")

for region in failure_config.regions:
    expected_reward = failure_info["local_rewards"][region] - failure_config.failure_penalty
    reward_difference = failure_rewards[region] - expected_reward
    check_validation(abs(reward_difference) < 1e-9, f"Region {region} receives the failure penalty")

print("Failure triggered for:", failure_info["failed_regions"])
print("Failure penalty applied to every regional reward.")

PASS: Pollution limit failure terminates the episode
PASS: Failure is not reported as a time-limit truncation
PASS: Failed regions and pollutants are recorded
PASS: Region 0 air reaches its limit
PASS: Region 1 air reaches its limit
PASS: Region 2 air reaches its limit
PASS: Region 3 air reaches its limit
PASS: Region 0 receives the failure penalty
PASS: Region 1 receives the failure penalty
PASS: Region 2 receives the failure penalty
PASS: Region 3 receives the failure penalty
Failure triggered for: ((0, 'air'), (1, 'air'), (2, 'air'), (3, 'air'))
Failure penalty applied to every regional reward.


## 6. Review the validation results

In [56]:
failed_checks = []

for status, description in validation_checks:
    if status == "FAIL":
        failed_checks.append(description)

print(f"Passed {len(validation_checks) - len(failed_checks)} of {len(validation_checks)} checks.")

if failed_checks:
    raise RuntimeError("Validation failed: " + ", ".join(failed_checks))

print("All environment validation checks passed.")

Passed 45 of 45 checks.
All environment validation checks passed.


## Interpretation

